# Hospital Readmission Prediction - Stage 2 Preprocessing

This notebook implements the Stage 2 preprocessing and feature engineering workflow based on the Stage 1 EDA findings. It prepares a clean dataset for modeling while preserving the patient-level grouping needed for a leak-free split.

## 1. Load data and carry forward Stage 1 decisions

Load the raw dataset, replace question marks with missing values, remove encounters ending in death, recreate the binary `readmitted_30` target, and drop patient identifiers from the feature matrix. `patient_nbr` is kept only as a grouping key for the split.

In [1]:
import json
from io import StringIO
from pathlib import Path
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import joblib

project_root = Path.cwd().resolve().parent if 'notebooks' in str(Path.cwd()) else Path.cwd().resolve()
raw_data_dir = project_root / 'diabetes+130-us+hospitals+for+years+1999-2008'
data_path = raw_data_dir / 'diabetic_data.csv'
mapping_path = raw_data_dir / 'IDS_mapping.csv'
figures_dir = project_root / 'results' / 'figures'
results_dir = project_root / 'results'
processed_dir = project_root / 'data' / 'processed'
figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

print('Loading dataset...')
df = pd.read_csv(data_path)
df = df.replace('?', np.nan)

# Confirm the mortality disposition codes in this project's UCI mapping file.
# IDS_mapping.csv contains several tables; isolate its discharge-disposition section.
mapping_lines = mapping_path.read_text(encoding='utf-8').splitlines()
disposition_header = next(i for i, line in enumerate(mapping_lines) if line.startswith('discharge_disposition_id,'))
disposition_end = next(i for i, line in enumerate(mapping_lines[disposition_header + 1:], disposition_header + 1) if line == ',')
disposition_mapping = pd.read_csv(StringIO('\n'.join(mapping_lines[disposition_header:disposition_end])))
expired_disposition_ids = disposition_mapping.loc[
    disposition_mapping['description'].str.contains('Expired', case=False, na=False),
    'discharge_disposition_id',
].astype(int).tolist()
expected_expired_ids = [11, 19, 20, 21]
assert expired_disposition_ids == expected_expired_ids, (
    f'Unexpected expired disposition mapping: {expired_disposition_ids}'
)
print('Verified expired disposition IDs from IDS_mapping.csv:', expired_disposition_ids)
print(disposition_mapping[disposition_mapping['discharge_disposition_id'].isin(expired_disposition_ids)].to_string(index=False))

# Create the target before filtering so its distribution can be compared fairly.
df['readmitted_30'] = df['readmitted'].apply(lambda x: 1 if str(x).strip() == '<30' else 0)
rows_before_filter = len(df)
class_balance_before = df['readmitted_30'].value_counts().sort_index()
class_rate_before = df['readmitted_30'].mean()

df = df.loc[~df['discharge_disposition_id'].isin(expired_disposition_ids)].copy()
rows_after_filter = len(df)
class_balance_after = df['readmitted_30'].value_counts().sort_index()
class_rate_after = df['readmitted_30'].mean()

print(f'Rows before expired-patient filter: {rows_before_filter:,}')
print(f'Rows after expired-patient filter:  {rows_after_filter:,}')
print(f'Rows removed: {rows_before_filter - rows_after_filter:,} ({(rows_before_filter - rows_after_filter) / rows_before_filter:.2%})')
print('Class balance before filter:')
print(class_balance_before)
print(f'Readmitted <30 rate before filter: {class_rate_before:.4%}')
print('Class balance after filter:')
print(class_balance_after)
print(f'Readmitted <30 rate after filter:  {class_rate_after:.4%}')

df = df.drop(columns=['encounter_id'], errors='ignore')
df = df.drop(columns=['readmitted'], errors='ignore')

print('Filtered shape:', df.shape)
print('Target counts after expired-patient filter:')
print(df['readmitted_30'].value_counts(dropna=False))


Loading dataset...


Verified expired disposition IDs from IDS_mapping.csv: [11, 19, 20, 21]
 discharge_disposition_id                                            description
                       11                                                Expired
                       19               Expired at home. Medicaid only, hospice.
                       20 Expired in a medical facility. Medicaid only, hospice.
                       21        Expired, place unknown. Medicaid only, hospice.
Rows before expired-patient filter: 101,766
Rows after expired-patient filter:  100,114
Rows removed: 1,652 (1.62%)
Class balance before filter:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64
Readmitted <30 rate before filter: 11.1599%
Class balance after filter:
readmitted_30
0    88757
1    11357
Name: count, dtype: int64
Readmitted <30 rate after filter:  11.3441%


Filtered shape: (100114, 49)
Target counts after expired-patient filter:
readmitted_30
0    88757
1    11357
Name: count, dtype: int64


## 2. Handle the near-unusable columns

Drop `weight`, `max_glu_serum`, and `A1Cresult` after converting test presence into binary flags. Impute the moderately missing categories as `Unknown` and keep small missingness groups using a default category.

In [2]:
df['had_glu_serum_test'] = df['max_glu_serum'].notna().astype(int)
df['had_A1C_test'] = df['A1Cresult'].notna().astype(int)

drop_columns = ['weight', 'max_glu_serum', 'A1Cresult']
df = df.drop(columns=[col for col in drop_columns if col in df.columns])

df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')
df['payer_code'] = df['payer_code'].fillna('Unknown')

for col in ['race', 'diag_1', 'diag_2', 'diag_3']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

missing_summary = df[['medical_specialty', 'payer_code', 'race', 'diag_1', 'diag_2', 'diag_3']].isna().mean() * 100
print('Post-imputation missing summary (%)')
print(missing_summary)


Post-imputation missing summary (%)
medical_specialty    0.0
payer_code           0.0
race                 0.0
diag_1               0.0
diag_2               0.0
diag_3               0.0
dtype: float64


## 3. Collapse the near-constant drug columns

Create a compact medication-change signal from the drug columns that are dominated by a single value, then drop those noisy columns. Keep only more informative medication features for the modeling pipeline.

In [3]:
near_constant_meds = [
    'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol',
    'troglitazone', 'tolazamide', 'examide', 'citoglipton',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]
near_constant_meds = [col for col in near_constant_meds if col in df.columns]

dominant_values = {col: df[col].mode(dropna=False)[0] for col in near_constant_meds}
df['num_med_changes'] = sum((df[col] != dominant_values[col]).astype(int) for col in near_constant_meds)

keep_medications = ['insulin', 'metformin']
keep_medications = [col for col in keep_medications if col in df.columns]

drop_meds = [col for col in near_constant_meds if col not in keep_medications]
df = df.drop(columns=drop_meds)

print('Near-constant medication columns dropped:')
print(drop_meds)
print('Remaining medication columns:')
print([col for col in keep_medications if col in df.columns])


Near-constant medication columns dropped:
['repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
Remaining medication columns:
['insulin', 'metformin']


## 4. Engineer utilization and diagnosis features

Create a single prior-utilization score and encode the high-cardinality diagnosis codes using frequency encoding.

In [4]:
df['total_visits_prior'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']

for col in ['diag_1', 'diag_2', 'diag_3']:
    if col in df.columns:
        freq_name = f'{col}_freq'
        freq = df[col].value_counts(normalize=True)
        df[freq_name] = df[col].map(freq).fillna(0.0)

print('Engineered features added: total_visits_prior, diag_1_freq, diag_2_freq, diag_3_freq')


Engineered features added: total_visits_prior, diag_1_freq, diag_2_freq, diag_3_freq


## 5. Encoding strategy

Ordinal-encode age bands, one-hot encode low-cardinality categorical features, and preserve the diagnosis frequency encodings.

In [5]:
age_levels = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)', '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
df['age_ordinal'] = pd.Categorical(df['age'], categories=age_levels, ordered=True).codes

feature_columns = [
    'age_ordinal', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_diagnoses', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'total_visits_prior', 'num_med_changes', 'had_glu_serum_test',
    'had_A1C_test', 'diag_1_freq', 'diag_2_freq', 'diag_3_freq'
]
categorical_features = [
    'gender', 'race', 'admission_type_id', 'discharge_disposition_id',
    'admission_source_id', 'payer_code', 'medical_specialty', 'change',
    'diabetesMed'
]
for col in ['insulin', 'metformin']:
    if col in df.columns:
        categorical_features.append(col)

numeric_features = [col for col in feature_columns if col in df.columns] + ['age_ordinal']
numeric_features = list(dict.fromkeys(numeric_features))
categorical_features = [col for col in categorical_features if col in df.columns] + ['age_ordinal']
categorical_features = [col for col in categorical_features if col not in numeric_features]  # avoid duplicates

print('Numeric features:')
print(numeric_features)
print('Categorical features:')
print(categorical_features)


Numeric features:
['age_ordinal', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_diagnoses', 'number_outpatient', 'number_emergency', 'number_inpatient', 'total_visits_prior', 'num_med_changes', 'had_glu_serum_test', 'had_A1C_test', 'diag_1_freq', 'diag_2_freq', 'diag_3_freq']
Categorical features:
['gender', 'race', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'medical_specialty', 'change', 'diabetesMed', 'insulin', 'metformin']


## 6. Group-aware split

Split patients into train/validation/test groups with zero overlap and preserve readmission stratification at the patient level.

In [6]:
patient_target = df.groupby('patient_nbr')['readmitted_30'].max().astype(int)
patients = patient_target.index.to_numpy()
patient_labels = patient_target.to_numpy()

sss_outer = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(sss_outer.split(patients, patient_labels))
train_val_patients = patients[train_val_idx]
test_patients = patients[test_idx]

sss_inner = StratifiedShuffleSplit(n_splits=1, test_size=0.17647058823529413, random_state=42)
train_idx, val_idx = next(sss_inner.split(train_val_patients, patient_target.loc[train_val_patients].to_numpy()))
train_patients = train_val_patients[train_idx]
val_patients = train_val_patients[val_idx]

train_mask = df['patient_nbr'].isin(train_patients)
val_mask = df['patient_nbr'].isin(val_patients)
test_mask = df['patient_nbr'].isin(test_patients)

X = df[feature_columns + categorical_features].copy()
X = X.loc[:, ~X.columns.duplicated()]
y = df['readmitted_30']

X_train = X[train_mask].reset_index(drop=True)
X_val = X[val_mask].reset_index(drop=True)
X_test = X[test_mask].reset_index(drop=True)
y_train = y[train_mask].reset_index(drop=True)
y_val = y[val_mask].reset_index(drop=True)
y_test = y[test_mask].reset_index(drop=True)

print('Train patients', len(train_patients))
print('Validation patients', len(val_patients))
print('Test patients', len(test_patients))
print('Patient overlap train/val:', len(set(train_patients) & set(val_patients)))
print('Patient overlap train/test:', len(set(train_patients) & set(test_patients)))
print('Patient overlap val/test:', len(set(val_patients) & set(test_patients)))


Train patients 49307
Validation patients 10566
Test patients 10566
Patient overlap train/val: 0
Patient overlap train/test: 0
Patient overlap val/test: 0


## 7. Fit preprocessing pipeline and apply SMOTE

Fit the preprocessing pipeline on training data only, transform all splits, and apply SMOTE to the training set after the split.

In [7]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

pipeline = Pipeline(steps=[('preprocessor', preprocessor)])
pipeline.fit(X_train, y_train)
joblib.dump(pipeline, results_dir / 'preprocessing_pipeline.joblib')

X_train_proc = pipeline.transform(X_train)
X_val_proc = pipeline.transform(X_val)
X_test_proc = pipeline.transform(X_test)
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

X_train_preSMOTE_df = pd.DataFrame(X_train_proc, columns=feature_names)
X_train_preSMOTE_df.to_parquet(processed_dir / 'X_train_preSMOTE.parquet', index=False)
y_train.to_frame('readmitted_30').to_parquet(processed_dir / 'y_train_preSMOTE.parquet', index=False)

print('Class balance before SMOTE (train):')
print(y_train.value_counts())
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train)
print('Class balance after SMOTE (train):')
print(pd.Series(y_train_res).value_counts())

feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
X_train_df = pd.DataFrame(X_train_res, columns=feature_names)
X_val_df = pd.DataFrame(X_val_proc, columns=feature_names)
X_test_df = pd.DataFrame(X_test_proc, columns=feature_names)

X_train_df.to_parquet(processed_dir / 'X_train.parquet', index=False)
X_val_df.to_parquet(processed_dir / 'X_val.parquet', index=False)
X_test_df.to_parquet(processed_dir / 'X_test.parquet', index=False)
pd.Series(y_train_res, name='readmitted_30').to_frame().to_parquet(processed_dir / 'y_train.parquet', index=False)
y_val.to_frame('readmitted_30').to_parquet(processed_dir / 'y_val.parquet', index=False)
y_test.to_frame('readmitted_30').to_parquet(processed_dir / 'y_test.parquet', index=False)

print('Saved processed datasets to', processed_dir)
print('Saved preprocessing pipeline to', results_dir / 'preprocessing_pipeline.joblib')


Class balance before SMOTE (train):
readmitted_30
0    62109
1     7957
Name: count, dtype: int64


C:\Users\sagir\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\sagir\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


Class balance after SMOTE (train):


readmitted_30
0    62109
1    62109
Name: count, dtype: int64


Saved processed datasets to C:\Users\sagir\OneDrive\Desktop\capstone\data\processed
Saved preprocessing pipeline to C:\Users\sagir\OneDrive\Desktop\capstone\results\preprocessing_pipeline.joblib
